In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

In [2]:
df = pd.read_csv("../../data/processed/diabetic_retinopathy/image_metadata.csv")

In [3]:
df

,id_code,height,width,diagnosis,file_path,blur_score,brightness
0,0024cdab0c1e,224,224,1,..\..\data\raw\diabetic_retinopathy\Mild\0024c...,262.95,66.78
1,00cb6555d108,224,224,1,..\..\data\raw\diabetic_retinopathy\Mild\00cb6...,238.39,65.31
2,0124dffecf29,224,224,1,..\..\data\raw\diabetic_retinopathy\Mild\0124d...,406.80,90.57
3,01b3aed3ed4c,224,224,1,..\..\data\raw\diabetic_retinopathy\Mild\01b3a...,252.08,77.22
4,0369f3efe69b,224,224,1,..\..\data\raw\diabetic_retinopathy\Mild\0369f...,287.19,59.32
...,...,...,...,...,...,...,...
3657,f9156aeffc5e,224,224,3,..\..\data\raw\diabetic_retinopathy\Severe\f91...,135.47,34.86
3658,fb61230b99dd,224,224,3,..\..\data\raw\diabetic_retinopathy\Severe\fb6...,133.34,54.09
3659,fcc6aa6755e6,224,224,3,..\..\data\raw\diabetic_retinopathy\Severe\fcc...,148.21,37.61
3660,fda39982a810,224,224,3,..\..\data\raw\diabetic_retinopathy\Severe\fda...,115.17,45.88


# train/validation/test split

In [4]:
from sklearn.model_selection import train_test_split

# train/test split
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["diagnosis"],
    random_state=10
)
# train/validation split
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["diagnosis"],
    random_state=10
)

len(train_df), len(val_df), len(test_df)

(2343, 586, 733)

In [5]:
len(train_df) + len(val_df) + len(test_df)

3662

### Percentage of each part

In [6]:
print((len(train_df) * 100) / 3662)
print((len(val_df) * 100) / 3662)
print((len(test_df) * 100) / 3662)

63.981430912069904
16.002184598580012
20.01638448935008


#### Checking torch version

In [7]:
# !pip install torch torchvision

In [8]:
import torch
torch.__version__

'2.13.0+cpu'

## Build a class for dataset

In [9]:
from torch.utils.data import Dataset
from PIL import Image


class RetinopathyDataset(Dataset):
    
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        # PIL Image (opencv is not supported here)
        image = Image.open(row["file_path"]).convert("RGB")
        
        label = row["diagnosis"]
        
        if self.transform:
            image = self.transform(image)

        return image, label
    

## Create transforms

#### Calculate Mean/Std RGB for train_df

##### import module

In [10]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]

sys.path.append(str(ROOT))

In [11]:
from shared.preprocessing.statistics import get_mean_std_rgb

mean, std = get_mean_std_rgb(train_df)
mean, std

(array([0.41221823, 0.21995935, 0.07291832]),
 array([0.27404434, 0.1498604 , 0.0806435 ]))

###### Mean RGB for the whole dataset: [0.41346971 0.22068593 0.07336238]
###### Std RGB for the whole dataset: [0.27443648 0.14982664 0.08072127]

In [12]:
from torchvision import transforms


train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1,
        saturation=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

## Create dataset

In [13]:
train_dataset = RetinopathyDataset(
    dataframe=train_df,
    transform=train_transform,
)

val_dataset = RetinopathyDataset(
    dataframe=val_df,
    transform=val_transform,
)

test_dataset = RetinopathyDataset(
    dataframe=test_df,
    transform=val_transform,
)


## Create Dataloader

##### Logical Processors

In [14]:
import os

print(os.cpu_count())

12


In [15]:
from torch.utils.data import DataLoader

num_workers = os.cpu_count()
pin_memory = torch.cuda.is_available()


train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory
)



In [16]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)
print(images.dtype)
print(labels.dtype)

torch.Size([32, 3, 224, 224])
torch.Size([32])
torch.float32
torch.int64


In [17]:
print(torch.unique(labels, return_counts=True))

(tensor([0, 1, 2, 3, 4]), tensor([16,  2, 11,  2,  1]))


##### Class Imbalance observed

## ResNet50 pretrained

#### Backbone will be frozen

In [18]:
from torch import nn
from torchvision.models import resnet50, ResNet50_Weights

def create_resnet50(num_classes: int):
    # Load pretrained ResNet50
    model = resnet50(
        weights=ResNet50_Weights.DEFAULT
    )
    # Freeze the pretrained layers
    for param in model.parameters():
        param.requires_grad = False

    # Replace the final classifier
    model.fc = nn.Linear(
        in_features=model.fc.in_features,
        out_features=num_classes
    )
    return model

In [19]:
model = create_resnet50(num_classes=5)
print(model.fc)

Linear(in_features=2048, out_features=5, bias=True)


In [20]:
outputs = model(images)

print("Input :", images.shape)
print("Labels:", labels.shape)
print("Output:", outputs.shape)

Input : torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])
Output: torch.Size([32, 5])


## Set device type

In [21]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cpu


#### Class weights

In [22]:
class_counts = train_df["diagnosis"].value_counts().sort_index()

print(class_counts)

diagnosis
0    1155
1     237
2     639
3     123
4     189
Name: count, dtype: int64


In [23]:
class_weights = len(train_df) / (len(class_counts) * class_counts)

class_weights = torch.tensor(
    class_weights.values,
    dtype=torch.float32
)
class_weights = class_weights.to(device)

print(class_weights)

tensor([0.4057, 1.9772, 0.7333, 3.8098, 2.4794])


In [24]:
len(class_counts) * class_counts


diagnosis
0    5775
1    1185
2    3195
3     615
4     945
Name: count, dtype: int64

In [25]:
from torch import nn


criterion = nn.CrossEntropyLoss(
    weight=class_weights
)
criterion

CrossEntropyLoss()

### Optimizer

In [26]:
from torch.optim import AdamW

optimizer = AdamW(
    params=model.parameters(),
    lr=1e-3,
    weight_decay=1e-3
)

In [27]:
model = model.to(device)

## Training Loop

### Train

In [28]:
def train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device
):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        # Move data to device
        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        # Accumulate loss
        running_loss += loss.item() * images.size(0)

        # Predictions
        predictions = outputs.argmax(dim=1)

        # Accuracy
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

### Validation

In [29]:
def validate_one_epoch(
    model,
    val_loader,
    criterion,
    device
):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():

        for images, labels in val_loader:

            # Move data to device
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)

            # Calculate loss
            loss = criterion(outputs, labels)

            # Accumulate loss
            running_loss += loss.item() * images.size(0)

            # Predictions
            predictions = outputs.argmax(dim=1)

            # Accuracy
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [30]:
num_epochs = 20
best_val_loss = float("inf")

checkpoint_path = "best_model.pth"
for epoch in range(num_epochs):

    train_loss, train_accuracy = train_one_epoch(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device
    )

    val_loss, val_accuracy = validate_one_epoch(
        model=model,
        val_loader=val_loader,
        criterion=criterion,
        device=device
    )

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_accuracy:.4f}"
    )

    # Save the best model
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            checkpoint_path
        )

        print("✓ Best model saved")

Epoch [1/20] Train Loss: 1.3833 Train Acc: 0.5578 Val Loss: 1.2389 Val Acc: 0.7014
✓ Best model saved
Epoch [2/20] Train Loss: 1.1601 Train Acc: 0.6829 Val Loss: 1.1641 Val Acc: 0.6382
✓ Best model saved
Epoch [3/20] Train Loss: 1.0604 Train Acc: 0.7089 Val Loss: 1.0877 Val Acc: 0.7201
✓ Best model saved
Epoch [4/20] Train Loss: 1.0065 Train Acc: 0.7162 Val Loss: 1.0660 Val Acc: 0.6587
✓ Best model saved
Epoch [5/20] Train Loss: 0.9758 Train Acc: 0.7397 Val Loss: 1.0267 Val Acc: 0.7065
✓ Best model saved
Epoch [6/20] Train Loss: 0.9191 Train Acc: 0.7358 Val Loss: 1.0417 Val Acc: 0.6980
Epoch [7/20] Train Loss: 0.9134 Train Acc: 0.7525 Val Loss: 1.0043 Val Acc: 0.7321
✓ Best model saved
Epoch [8/20] Train Loss: 0.8866 Train Acc: 0.7431 Val Loss: 1.0142 Val Acc: 0.6843
Epoch [9/20] Train Loss: 0.8587 Train Acc: 0.7589 Val Loss: 0.9865 Val Acc: 0.7218
✓ Best model saved
Epoch [10/20] Train Loss: 0.8371 Train Acc: 0.7580 Val Loss: 1.0264 Val Acc: 0.7218
Epoch [11/20] Train Loss: 0.8465 Tra

#### We can add a Scheduler for tuning Learning Rate. But because i ran it in cpu, it took a lot of time so you can try it with Scheduler too!

## Load Model

In [31]:
model.load_state_dict(
    torch.load(
        checkpoint_path,
        map_location=device
    )
)

model = model.to(device)

In [33]:
model.fc

Linear(in_features=2048, out_features=5, bias=True)